In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


# ============================================================
# IMPORTS
# ============================================================


import json


from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import timm

from transformers import (
    AutoTokenizer,
    AutoModel
)

# ============================================================
# CONFIG
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 224
EMBED_DIM = 256
BATCH_SIZE = 8
EPOCHS = 5
LR = 1e-4

# ------------------------------------------------------------
# CHANGE THIS FOR YOUR KAGGLE DATASET
# ------------------------------------------------------------

DATASET_ROOT = "/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset"

PATIENTS_CSV = f"{DATASET_ROOT}/patients.csv"
VISITS_CSV = f"{DATASET_ROOT}/visits.csv"
REPORTS_JSONL = f"{DATASET_ROOT}/diagnosis_reports.jsonl"
BIOPSY_NPY = f"{DATASET_ROOT}/liquid_biopsy_vectors.npy"
SCANS_ROOT = f"{DATASET_ROOT}/ct_scans"

REPORT_MODEL = "emilyalsentzer/Bio_ClinicalBERT"

# ============================================================
# DATASET
# ============================================================

class LungCancerDataset(Dataset):

    def __init__(
        self,
        patients_csv,
        visits_csv,
        reports_jsonl,
        biopsy_npy,
        scans_root
    ):

        self.patients = pd.read_csv(patients_csv)
        self.visits = pd.read_csv(visits_csv)

        self.scans_root = Path(scans_root)

        # ----------------------------------------------------
        # REPORTS
        # ----------------------------------------------------

        with open(reports_jsonl) as f:
            reports = [json.loads(x) for x in f]

        self.reports = {
            (r["patient_id"], r["visit"]): r["report"]
            for r in reports
        }

        # ----------------------------------------------------
        # BIOPSY
        # ----------------------------------------------------

        self.biopsy = np.load(
            biopsy_npy,
            allow_pickle=True
        ).item()

        self.samples = []

        # ----------------------------------------------------
        # BUILD SAMPLES
        # ----------------------------------------------------

        for _, row in self.visits.iterrows():

            pid = row["patient_id"]

            patient_row = self.patients[
                self.patients.patient_id == pid
            ].iloc[0]

            self.samples.append({

                "patient_id": pid,

                "visit": int(row["visit"]),

                "scan_file": row["scan_file"],

                "label": float(
                    patient_row.eventual_cancer
                )
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        s = self.samples[idx]

        # ====================================================
        # LOAD CT SCAN
        # ====================================================

        scan_path = (
            self.scans_root /
            Path(s["scan_file"]).name
        )

        scan = np.load(scan_path)

        # use middle slice
        scan = scan[scan.shape[0] // 2]

        scan = torch.tensor(scan).float()

        scan = scan.unsqueeze(0)

        scan = F.interpolate(
            scan.unsqueeze(0),
            size=(IMG_SIZE, IMG_SIZE),
            mode="bilinear"
        ).squeeze(0)

        scan = scan.repeat(3, 1, 1)

        # ====================================================
        # LOAD BIOPSY VECTOR
        # ====================================================

        biopsy = self.biopsy[
            s["patient_id"]
        ][s["visit"] - 1]

        biopsy = torch.tensor(
            biopsy
        ).float()

        # ====================================================
        # LOAD REPORT
        # ====================================================

        report = self.reports[
            (s["patient_id"], s["visit"])
        ]

        # ====================================================
        # LABEL
        # ====================================================

        label = torch.tensor(
            [s["label"]]
        ).float()

        return {

            "image": scan,

            "biopsy": biopsy,

            "report": report,

            "label": label
        }

# ============================================================
# CNN BRANCH
# ============================================================

class CNNBranch(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = timm.create_model(
            "resnet34",
            pretrained=True,
            num_classes=0
        )

        self.proj = nn.Linear(
            self.backbone.num_features,
            EMBED_DIM
        )

    def forward(self, x):

        x = self.backbone(x)

        x = self.proj(x)

        return x

# ============================================================
# SWIN BRANCH
# ============================================================

class SwinBranch(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=True,
            num_classes=0
        )

        self.proj = nn.Linear(
            self.backbone.num_features,
            EMBED_DIM
        )

    def forward(self, x):

        x = self.backbone(x)

        x = self.proj(x)

        return x

# ============================================================
# HYBRID IMAGE ENCODER
# ============================================================

class HybridImageEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.cnn = CNNBranch()

        self.swin = SwinBranch()

        self.fusion = nn.Sequential(

            nn.Linear(
                EMBED_DIM * 2,
                EMBED_DIM
            ),

            nn.ReLU(),

            nn.Dropout(0.2)
        )

    def forward(self, x):

        cnn_feat = self.cnn(x)

        swin_feat = self.swin(x)

        feat = torch.cat(
            [cnn_feat, swin_feat],
            dim=-1
        )

        feat = self.fusion(feat)

        return feat

# ============================================================
# LIQUID BIOPSY ENCODER
# ============================================================

class LiquidBiopsyEncoder(nn.Module):

    def __init__(self, input_dim=128):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 256),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(256, EMBED_DIM),

            nn.ReLU()
        )

    def forward(self, x):

        return self.net(x)

# ============================================================
# REPORT ENCODER
# ============================================================

class ReportEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # TOKENIZER + BERT MODEL
        # ----------------------------------------------------

        self.tokenizer = AutoTokenizer.from_pretrained(
            REPORT_MODEL
        )

        self.model = AutoModel.from_pretrained(
            REPORT_MODEL
        )

        # freeze BERT weights to use as embeddings
        for param in self.model.parameters():
            param.requires_grad = False

        self.proj = nn.Sequential(

            nn.Linear(
                self.model.config.hidden_size,
                EMBED_DIM
            ),

            nn.ReLU(),

            nn.Dropout(0.2)
        )

    def mean_pooling(
        self,
        model_output,
        attention_mask
    ):

        token_embeddings = model_output.last_hidden_state

        input_mask_expanded = (
            attention_mask
            .unsqueeze(-1)
            .expand(token_embeddings.size())
            .float()
        )

        return torch.sum(
            token_embeddings * input_mask_expanded,
            dim=1
        ) / torch.clamp(
            input_mask_expanded.sum(dim=1),
            min=1e-9
        )

    def forward(self, reports):

        encoded = self.tokenizer(
            reports,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        encoded = {
            k: v.to(DEVICE)
            for k, v in encoded.items()
        }

        # ----------------------------------------------------
        # EXTRACT BERT EMBEDDINGS
        # ----------------------------------------------------

        with torch.no_grad():

            outputs = self.model(**encoded)

            # mean pooled sentence embedding
            embeddings = self.mean_pooling(
                outputs,
                encoded["attention_mask"]
            )

        embeddings = self.proj(embeddings)

        return embeddings

# ============================================================
# EXPERT FUSION
# ============================================================

class ExpertFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.gating = nn.Sequential(

            nn.Linear(
                EMBED_DIM * 3,
                256
            ),

            nn.ReLU(),

            nn.Linear(256, 3),

            nn.Softmax(dim=-1)
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                EMBED_DIM,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256, 1)
        )

    def forward(
        self,
        image_feat,
        biopsy_feat,
        report_feat
    ):

        concat = torch.cat([
            image_feat,
            biopsy_feat,
            report_feat
        ], dim=-1)

        gates = self.gating(concat)

        fused = (
            gates[:, 0:1] * image_feat +
            gates[:, 1:2] * biopsy_feat +
            gates[:, 2:3] * report_feat
        )

        risk = self.classifier(fused)

        return risk, fused, gates

# ============================================================
# COMPLETE MODEL
# ============================================================

class MultimodalCancerRiskModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.image_encoder = HybridImageEncoder()

        self.biopsy_encoder = LiquidBiopsyEncoder()

        self.report_encoder = ReportEncoder()

        self.fusion = ExpertFusion()

    def forward(
        self,
        images,
        biopsy,
        reports
    ):

        image_feat = self.image_encoder(images)

        biopsy_feat = self.biopsy_encoder(biopsy)

        report_feat = self.report_encoder(reports)

        risk, embedding, gates = self.fusion(
            image_feat,
            biopsy_feat,
            report_feat
        )

        return {

            "risk": risk,

            "embedding": embedding,

            "gates": gates
        }

# ============================================================
# TRAIN LOOP
# ============================================================

def train_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    total_loss = 0

    for batch in loader:

        images = batch["image"].to(DEVICE)

        biopsy = batch["biopsy"].to(DEVICE)

        reports = batch["report"]

        labels = batch["label"].to(DEVICE)

        out = model(
            images,
            biopsy,
            reports
        )

        loss = criterion(
            out["risk"],
            labels
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    print("Loading dataset...")

    dataset = LungCancerDataset(

        patients_csv=PATIENTS_CSV,

        visits_csv=VISITS_CSV,

        reports_jsonl=REPORTS_JSONL,

        biopsy_npy=BIOPSY_NPY,

        scans_root=SCANS_ROOT
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2
    )

    print("Building model...")

    model = MultimodalCancerRiskModel().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR
    )

    criterion = nn.BCEWithLogitsLoss()

    print("Training...")

    for epoch in range(EPOCHS):

        loss = train_epoch(
            model,
            loader,
            optimizer,
            criterion
        )

        print(
            f"Epoch {epoch+1}/{EPOCHS} "
            f"Loss: {loss:.4f}"
        )

    # ========================================================
    # SAVE MODEL
    # ========================================================

    torch.save(
        model.state_dict(),
        "/kaggle/working/multimodal_cancer_model.pt"
    )

    print()

    print("Training complete.")

    print(
        "Model saved to:"
    )

    print(
        "/kaggle/working/multimodal_cancer_model.pt"
    )

/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/visits.csv
/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/patients.csv
/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/liquid_biopsy_vectors.npy
/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/diagnosis_reports.jsonl
/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/ct_scans/P0002_v3.npy
/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/ct_scans/P0489_v3.npy
/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/ct_scans/P0048_v1.npy
/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/ct_scans/P0359_v1.npy
/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/ct_scans/P0156_v1.npy
/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/ct_scans/P0270_v3.npy
/kaggle/input/datasets/allistermyth/openai/synthetic_lung_dataset/ct_scans/P0383_v3.npy
/kaggle/input/datasets/allistermyth/openai/syn

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training...
Epoch 1/2 Loss: 0.4716
Epoch 2/2 Loss: 0.4611

Training complete.
Model saved to:
/kaggle/working/multimodal_cancer_model.pt
